# Question 9 
In this exercise, we will predict the number of applications received using the other variables in the College data set.

### (a)  
Split the data set into a training set and a test set.

In [20]:
# Import necessary libraries
from ISLP import load_data
import numpy as np
from sklearn.model_selection import train_test_split

# Load the College dataset
College = load_data('College')
College

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,No,2197,1515,543,4,26,3089,2029,6797,3900,500,1200,60,60,21.0,14,4469,40
773,Yes,1959,1805,695,24,47,2849,1107,11520,4960,600,1250,73,75,13.3,31,9189,83
774,Yes,2097,1915,695,34,61,2793,166,6900,4200,617,781,67,75,14.4,20,8323,49
775,Yes,10705,2453,1317,95,99,5217,83,19840,6510,630,2115,96,96,5.8,49,40386,99


In [21]:
# Split the data into predictors (X) and response (y)
X = College.drop(columns=['Apps'])  # The response variable is 'Apps' for number of applications
y = College['Apps']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the sizes of the training and test sets
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 621 samples
Test set size: 156 samples


### (b)  
Fit a linear model using least squares on the training set, and report the test error obtained.

In [22]:
# Fit a linear model using least squares and report test error
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

# One-hot encode categorical variables and align train/test columns
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# Fit linear regression
lm = LinearRegression()
lm.fit(X_train_enc, y_train)

# Predict and compute test MSE and RMSE
y_pred = lm.predict(X_test_enc)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

print(f"Test MSE: {mse:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# Optionally show a few coefficients
coef_df = pd.Series(lm.coef_, index=X_train_enc.columns).sort_values(key=abs, ascending=False)
print('\nTop 8 coefficients (by absolute value):')
print(coef_df.head(8))

Test MSE: 1492443.3790
Test RMSE: 1221.6560

Top 8 coefficients (by absolute value):
Private_Yes   -651.606978
Top10perc       51.706571
Top25perc      -14.992375
PhD            -10.422542
Grad.Rate        8.563253
S.F.Ratio        4.944358
Accept           1.665580
Terminal        -1.406325
dtype: float64


### (c)  
Fit a ridge regression model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained.

In [23]:
# Data preparation: encode and scale features
from sklearn.preprocessing import StandardScaler
import pandas as pd

# One-hot encode categorical variables and align train/test columns
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# Standardize the features for regularization models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

In [24]:
# Fit a Ridge regression with lambda selected by cross-validation
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Use RidgeCV with 10-fold CV and MSE scoring
alphas = np.logspace(-6, 6, 25)
ridge_cv = RidgeCV(alphas=alphas, cv=10, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)
best_alpha = ridge_cv.alpha_

# Refit Ridge with the chosen alpha on the full training set
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train_scaled, y_train)

# Predict on test set and compute error
y_ridge_pred = ridge.predict(X_test_scaled)
mse_ridge = mean_squared_error(y_test, y_ridge_pred)
rmse_ridge = mse_ridge ** 0.5

print(f"Best alpha (lambda): {best_alpha}")
print(f"Ridge Test MSE: {mse_ridge:.4f}")
print(f"Ridge Test RMSE: {rmse_ridge:.4f}")

coef_ridge = pd.Series(ridge.coef_, index=X_train_enc.columns).sort_values(key=abs, ascending=False)
print('\nTop 10 Ridge coefficients (by absolute value, on scaled features):')
print(coef_ridge.head(10))

Best alpha (lambda): 1e-06
Ridge Test MSE: 1492443.3631
Ridge Test RMSE: 1221.6560

Top 10 Ridge coefficients (by absolute value, on scaled features):
Accept         4103.246143
Top10perc       910.378909
Enroll         -903.130392
Top25perc      -295.744241
Private_Yes    -287.818376
Outstate       -285.927523
F.Undergrad     257.672149
Expend          216.757440
Room.Board      175.589730
PhD            -171.368180
dtype: float64


### (d)  
Fit a lasso model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [25]:
# Fit a Lasso regression with lambda selected by cross-validation
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Define candidate alphas (lambdas)
alphas = np.logspace(-6, 6, 25)

# Use LassoCV with 10-fold CV and MSE scoring
lasso_cv = LassoCV(alphas=alphas, cv=10, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

# Best alpha chosen by CV
best_alpha_lasso = lasso_cv.alpha_

# Refit Lasso with the chosen alpha on the full training set
lasso = Lasso(alpha=best_alpha_lasso)
lasso.fit(X_train_scaled, y_train)

# Predict on test set and compute error
y_lasso_pred = lasso.predict(X_test_scaled)
mse_lasso = mean_squared_error(y_test, y_lasso_pred)
rmse_lasso = mse_lasso ** 0.5

# Number of non-zero coefficients
num_nonzero = np.sum(lasso.coef_ != 0)

print(f"Best alpha (lambda): {best_alpha_lasso}")
print(f"Lasso Test MSE: {mse_lasso:.4f}")
print(f"Lasso Test RMSE: {rmse_lasso:.4f}")
print(f"Number of non-zero coefficients: {num_nonzero}")

# Show top coefficients by absolute value (non-zero only)
coef_lasso = pd.Series(lasso.coef_, index=X_train_enc.columns)
nonzero_coef = coef_lasso[coef_lasso != 0].sort_values(key=abs, ascending=False)
print('\nTop 10 non-zero Lasso coefficients (by absolute value, on scaled features):')
print(nonzero_coef.head(10))

Best alpha (lambda): 1e-06
Lasso Test MSE: 1492443.3748
Lasso Test RMSE: 1221.6560
Number of non-zero coefficients: 17

Top 10 non-zero Lasso coefficients (by absolute value, on scaled features):
Accept         4103.246185
Top10perc       910.378917
Enroll         -903.130426
Top25perc      -295.744244
Private_Yes    -287.818375
Outstate       -285.927525
F.Undergrad     257.672142
Expend          216.757434
Room.Board      175.589725
PhD            -171.368180
dtype: float64


### (e)  
Fit a PCR model on the training set, with $M$ chosen by cross-validation. Report the test error obtained, along with the value of $M$ selected by cross-validation.

In [26]:
# Fit a PCR model with M chosen by cross-validation
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Define pipeline: PCA + Linear Regression
pipe = Pipeline([
    ('pca', PCA()),
    ('lr', LinearRegression())
])

# Parameter grid for M (number of components)
param_grid = {'pca__n_components': np.arange(1, X_train_scaled.shape[1] + 1)}

# Grid search with 10-fold CV
grid = GridSearchCV(pipe, param_grid, cv=10, scoring='neg_mean_squared_error')
grid.fit(X_train_scaled, y_train)

# Best M selected by CV
best_M = grid.best_params_['pca__n_components']

# Refit PCR with best M on full training set
pcr = Pipeline([
    ('pca', PCA(n_components=best_M)),
    ('lr', LinearRegression())
])
pcr.fit(X_train_scaled, y_train)

# Predict on test set and compute error
y_pcr_pred = pcr.predict(X_test_scaled)
mse_pcr = mean_squared_error(y_test, y_pcr_pred)
rmse_pcr = mse_pcr ** 0.5

print(f"Best M (number of components): {best_M}")
print(f"PCR Test MSE: {mse_pcr:.4f}")
print(f"PCR Test RMSE: {rmse_pcr:.4f}")

Best M (number of components): 17
PCR Test MSE: 1492443.3790
PCR Test RMSE: 1221.6560
